<a href="https://colab.research.google.com/github/dklishta/python-ai-Gailunaite-Darya/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных о чайных и кофейных компаниях

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas на примере исторических и финансовых показателей компаний.

**Данные:**
- [`tea_coffee_info.csv`](https://github.com/dklishta/python-ai-Gailunaite-Darya/blob/main/data/tea_coffee_info.csv) — информация о компаниях: название, тип продукта, страна, штаб-квартира, координаты, год основания
- [`tea_coffee_fin.csv`](https://github.com/dklishta/python-ai-Gailunaite-Darya/blob/main/data/tea_coffee_fin.csv) — финансовые показатели: выручка, количество сотрудников, чистая прибыль (с указанием года и валюты)

**Что мы делаем:**
1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файлы в pandas DataFrame
3. Очищаем и переименовываем столбцы (сохраняя ключи для объединения!)
4. Парсим координаты для будущей визуализации на карте
5. Смотрим структуру данных и делаем быструю валидацию
6. Анализируем географическое распределение и финансовые показатели

> ⚠️ **Важно**: Мы не удаляем технические идентификаторы Викиданных — они понадобятся для объединения таблиц в будущем.

## 🐱 [1] Клонируем репозиторий курса в Colab

In [4]:
# 🐱 Шаг 1. Клонируем ваш репозиторий курса в Colab

import os

repo = "python-ai-Gailunaite-Darya"  # ← ИЗМЕНЕНО: ваш репозиторий
repo_path = f"/content/{repo}"

if not os.path.exists(repo_path):
    !git clone -q https://github.com/dklishta/python-ai-Gailunaite-Darya.git  # ← ИЗМЕНЕНО: ваш URL

if os.getcwd() != repo_path:
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

/content/python-ai-Gailunaite-Darya
✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Gailunaite-Darya


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [5]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

import pandas as pd

# Читаем файл с общей информацией
df_info = pd.read_csv("data/tea_coffee_info.csv")
print("✅ Загружено строк в df_info:", len(df_info))

# Читаем файл с финансовыми данными
df_fin = pd.read_csv("data/tea_coffee_fin.csv")
print("✅ Загружено строк в df_fin:", len(df_fin))

# Показываем столбцы для проверки
print("\n📋 Столбцы df_info:", list(df_info.columns))
print("📋 Столбцы df_fin:", list(df_fin.columns))

✅ Загружено строк в df_info: 289
✅ Загружено строк в df_fin: 1555

📋 Столбцы df_info: ['company', 'companyLabel', 'productTypeLabel', 'countryLabel', 'hqLabel', 'hqCoord', 'foundingYearShort']
📋 Столбцы df_fin: ['company', 'companyLabel', 'productTypeLabel', 'revenue', 'revenueYear', 'revenueCurrencyLabel', 'employees', 'employeesYear', 'netProfit', 'netProfitYear', 'netProfitCurrencyLabel']


## 🧹 [2B] Очистка и переименование столбцов

### 🔧 Что делаем с `tea_coffee_info.csv`:
- **Переименовываем** `company` → `company_url` (сохраняем ключ Викиданных для будущего merge!)
- **Переименовываем** столбцы с суффиксом `Label`:
  - `companyLabel` → `company`
  - `productTypeLabel` → `productType`
  - `countryLabel` → `country`
  - `hqLabel` → `hq`
- **Парсим координаты** из `hqCoord` (текст вида `Point(-0.1275 51.5072)`) → отдельные столбцы `lon`, `lat` (числа)
- **Приводим** `foundingYearShort` к числовому типу и **фильтруем** значения ≤ 0 (это пропуски, а не годы)

### 🔧 Что делаем с `tea_coffee_fin.csv`:
- **Переименовываем** `company` → `company_url` (ключ для объединения!)
- **Переименовываем** `*Label` → короткие имена
- **Приводим** финансовые показатели к числовому типу
- **Обрабатываем дубликаты**: одна компания может иметь несколько строк (данные за разные годы)

> ⚠️ **Важно**: Столбец `company_url` — это единственный ключ, по которому мы сможем объединить `info` и `fin` через `pd.merge()`. Не удаляем его!

In [6]:
# 🧹 Шаг 2B. Очистка и переименование для tea_coffee_info.csv
# (❗ КРИТИЧЕСКИЕ ИСПРАВЛЕНИЯ по комментариям преподавателя)

import pandas as pd
import numpy as np

print("🧹 ОЧИСТКА: tea_coffee_info.csv")
print("=" * 70)

# ============================================================
# 🔑 1) НЕ удаляем company, а переименовываем в company_url
#    (это ключ для будущего merge с финансовыми данными!)
# ============================================================

if "company" in df_info.columns:
    df_info = df_info.rename(columns={"company": "company_url"})  # ← ИЗМЕНЕНО: переименование вместо удаления
    print("✅ Столбец 'company' переименован в 'company_url' (ключ для merge)")
else:
    print("⏭️  Столбец 'company' не найден")

# ============================================================
# 🏷️ 2) Переименовываем столбцы: убираем суффикс Label
# ============================================================

rename_map = {
    "companyLabel": "company",
    "productTypeLabel": "productType",
    "countryLabel": "country",
    "hqLabel": "hq",
}
df_info = df_info.rename(columns={k: v for k, v in rename_map.items() if k in df_info.columns})
print("✅ Столбцы *Label переименованы:", list(df_info.columns))

# ============================================================
# 🗺️ 3) ❗ НОВОЕ: Парсим координаты из hqCoord → lon, lat
#    Формат: "Point(-0.1275 51.507222)" → два числовых столбца
# ============================================================

if "hqCoord" in df_info.columns and df_info["hqCoord"].notna().any():
    # Извлекаем долготу и широту с помощью регулярного выражения
    coords = df_info["hqCoord"].str.extract(r'Point\(([^ ]+) ([^ )]+)\)')
    coords.columns = ["lon", "lat"]
    coords = coords.astype(float)  # Преобразуем в числа

    # Добавляем новые столбцы в DataFrame
    df_info = pd.concat([df_info, coords], axis=1)
    print("✅ Координаты распарсены: добавлены столбцы 'lon' и 'lat'")
else:
    print("⏭️  Столбец hqCoord пуст или отсутствует, пропускаем парсинг")

# ============================================================
# 🔢 4) Приводим foundingYearShort к числовому типу
# ============================================================

if "foundingYearShort" in df_info.columns:
    df_info["foundingYearShort"] = pd.to_numeric(
        df_info["foundingYearShort"], errors="coerce"
    )
    print("✅ foundingYearShort приведён к числовому типу")

# ============================================================
# 🚫 5) ❗ НОВОЕ: Фильтруем некорректные годы (0 и отрицательные)
#    0 — это не год основания, а пропуск, который превратился в число
# ============================================================

if "foundingYearShort" in df_info.columns:
    before = len(df_info)
    df_info = df_info[df_info["foundingYearShort"] > 0]  # ← ИЗМЕНЕНО: фильтр фейковых лет
    after = len(df_info)
    removed = before - after
    if removed > 0:
        print(f"⚠️  Удалено {removed} строк с некорректным годом основания (≤ 0)")
    else:
        print("✅ Все значения foundingYearShort корректны (> 0)")

# ============================================================
# 📊 6) Краткий отчёт о результате
# ============================================================

print(f"\n📋 Итоговый размер df_info: {df_info.shape[0]} строк, {df_info.shape[1]} столбцов")
print("📋 Столбцы:", list(df_info.columns))

# Проверяем, что ключ для merge на месте
if "company_url" in df_info.columns:
    print("✅ Ключ 'company_url' сохранён — готов к объединению с финансовыми данными")

🧹 ОЧИСТКА: tea_coffee_info.csv
✅ Столбец 'company' переименован в 'company_url' (ключ для merge)
✅ Столбцы *Label переименованы: ['company_url', 'company', 'productType', 'country', 'hq', 'hqCoord', 'foundingYearShort']
✅ Координаты распарсены: добавлены столбцы 'lon' и 'lat'
✅ foundingYearShort приведён к числовому типу
⚠️  Удалено 49 строк с некорректным годом основания (≤ 0)

📋 Итоговый размер df_info: 240 строк, 9 столбцов
📋 Столбцы: ['company_url', 'company', 'productType', 'country', 'hq', 'hqCoord', 'foundingYearShort', 'lon', 'lat']
✅ Ключ 'company_url' сохранён — готов к объединению с финансовыми данными


In [7]:
# 🧹 Шаг 2B.2. Очистка для tea_coffee_fin.csv
# (❗ КРИТИЧЕСКИЕ ИСПРАВЛЕНИЯ: ключ merge + обработка дубликатов)

print("\n🧹 ОЧИСТКА: tea_coffee_fin.csv")
print("=" * 70)

# ============================================================
# 🔑 1) Переименовываем company → company_url (ключ для merge!)
# ============================================================

if "company" in df_fin.columns:
    df_fin = df_fin.rename(columns={"company": "company_url"})
    print("✅ Столбец 'company' переименован в 'company_url'")

# ============================================================
# 🏷️ 2) Переименовываем *Label → короткие имена
# ============================================================

rename_map = {
    "companyLabel": "company",
    "productTypeLabel": "productType",
    "revenueCurrencyLabel": "revenueCurrency",
    "netProfitCurrencyLabel": "netProfitCurrency",
}
df_fin = df_fin.rename(columns={k: v for k, v in rename_map.items() if k in df_fin.columns})
print("✅ Столбцы *Label переименованы")

# ============================================================
# 🔢 3) Приводим финансовые показатели к числовому типу
# ============================================================

numeric_cols = ["revenue", "revenueYear", "employees", "employeesYear", "netProfit", "netProfitYear"]
for col in numeric_cols:
    if col in df_fin.columns:
        df_fin[col] = pd.to_numeric(df_fin[col], errors="coerce")

# ============================================================
# 🔄 4) ❗ НОВОЕ: Обработка дубликатов (декартово произведение)
#    Одна компания может иметь много строк (данные за разные годы).
#    Для рейтинга берём строку с ПОСЛЕДНИМ доступным годом выручки.
# ============================================================

if "revenue" in df_fin.columns and "revenueYear" in df_fin.columns:
    # Фильтруем строки с данными по выручке
    with_rev = df_fin[df_fin["revenue"].notna()].copy()

    if len(with_rev) > 0:
        # Сортируем по году и берём последнюю запись для каждой компании
        rev_latest = (
            with_rev
            .sort_values("revenueYear")
            .groupby("company_url", as_index=False)
            .last()[["company_url", "company", "productType", "revenue", "revenueYear", "revenueCurrency"]]
        )
        print(f"✅ Обработано {len(rev_latest)} уникальных компаний с данными по выручке")
        # Сохраняем для дальнейшего использования
        df_fin_revenue = rev_latest
    else:
        print("⚠️  Нет данных по выручке для обработки дубликатов")
        df_fin_revenue = df_fin.copy()
else:
    df_fin_revenue = df_fin.copy()

# ============================================================
# 📊 5) Честный отчёт о диспропорции чай/кофе
# ============================================================

print("\n📊 ДИСПРОПОРЦИЯ ДАННЫХ: чай vs кофе")
print("-" * 70)
if "productType" in df_fin.columns:
    product_counts = df_fin["productType"].value_counts()
    for prod, count in product_counts.items():
        pct = count / len(df_fin) * 100
        print(f"• {prod}: {count} строк ({pct:.1f}%)")

    tea_count = (df_fin["productType"] == "чай").sum()
    coffee_count = (df_fin["productType"] == "кофе").sum()

    if tea_count < 20:
        print(f"\n⚠️  ВНИМАНИЕ: чайных компаний с финансовыми данными очень мало ({tea_count} строк).")
        print("   Финансовый анализ будет репрезентативен только для кофейных компаний.")


🧹 ОЧИСТКА: tea_coffee_fin.csv
✅ Столбец 'company' переименован в 'company_url'
✅ Столбцы *Label переименованы
✅ Обработано 14 уникальных компаний с данными по выручке

📊 ДИСПРОПОРЦИЯ ДАННЫХ: чай vs кофе
----------------------------------------------------------------------
• кофе: 1457 строк (93.7%)
• чай: 98 строк (6.3%)


## 🔍 [3] Обзор данных: структура и первые строки

### 📋 Что смотрим для `df_info`:
- Размер таблицы, список столбцов (включая новые `lon`, `lat`);
- Первые строки с названиями компаний, странами, координатами;
- Статистику по `foundingYearShort` (после фильтрации > 0);
- Распределение по `productType` (чай / кофе) и `country`.

### 📋 Что смотрим для `df_fin`:
- Заполненность финансовых показателей (% реальных значений);
- Уникальные компании и типы продуктов;
- Диапазон годов (`revenueYear`, `netProfitYear`);
- Валюты, в которых представлены данные.

### 🧮 Методы pandas, которые мы используем:
| Метод | Зачем нужен |
|-------|-------------|
| `describe()` | Базовая статистика по числовым столбцам |
| `value_counts()` | Частота категорий (страны, продукты) |
| `notna().sum()` | Количество заполненных значений |
| `groupby() + last()` | Выбор последней записи по компании (для рейтинга) |

> 💡 **Подсказка**: Если в `df_fin` мало чайных компаний — это особенность Викиданных, а не ошибка кода. Мы честно отразим это в анализе.

In [8]:
# 🔍 Шаг 3. Обзор данных

def show_info(df, name, n=3):
    """Универсальный обзор DataFrame."""
    print(f"\n📊 {name}")
    print("=" * 70)
    print(f"Размер: {df.shape[0]} строк × {df.shape[1]} столбцов")
    print("Столбцы:", ", ".join(df.columns))
    print(f"\n📋 Первые {n} строки:")
    display(df.head(n))

    # Статистика по числам
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        print("\n📈 Статистика по числовым столбцам:")
        display(df[num_cols].describe().round(2))

    # Категории
    print("\n🏷️ Категориальные столбцы:")
    for col in df.select_dtypes(include=['object']).columns:
        if df[col].notna().any():
            unique = df[col].nunique()
            print(f"• {col}: {unique} уникальных")

# Обзор информационного файла
show_info(df_info, "🫖 df_info: общая информация о компаниях")

# Обзор финансового файла
show_info(df_fin, "💰 df_fin: финансовые показатели")

# ============================================================
# 💡 Дополнительно: проверка ключей для будущего merge
# ============================================================

print("\n" + "=" * 70)
print("🔑 ПРОВЕРКА КЛЮЧЕЙ ДЛЯ ОБЪЕДИНЕНИЯ ТАБЛИЦ")
print("=" * 70)

if "company_url" in df_info.columns and "company_url" in df_fin.columns:
    info_keys = df_info["company_url"].nunique()
    fin_keys = df_fin["company_url"].nunique()
    common = len(set(df_info["company_url"]) & set(df_fin["company_url"]))

    print(f"• Уникальных company_url в df_info: {info_keys}")
    print(f"• Уникальных company_url в df_fin: {fin_keys}")
    print(f"• Пересекающихся ключей: {common}")

    if common > 0:
        print("✅ Ключи совпадают — таблицы можно объединить через pd.merge()")
    else:
        print("⚠️  Нет пересекающихся ключей — проверьте данные")


📊 🫖 df_info: общая информация о компаниях
Размер: 240 строк × 9 столбцов
Столбцы: company_url, company, productType, country, hq, hqCoord, foundingYearShort, lon, lat

📋 Первые 3 строки:


,company_url,company,productType,country,hq,hqCoord,foundingYearShort,lon,lat
0,http://www.wikidata.org/entity/Q83164,Британская Ост-Индская компания,чай,Соединённое королевство Великобритании и Ирландии,Лондон,Point(-0.1275 51.507222222),1600.0,-0.127500,51.507222
1,http://www.wikidata.org/entity/Q83164,Британская Ост-Индская компания,чай,Соединённое королевство Великобритании и Ирландии,Лондон,Point(-0.1275 51.507222222),1600.0,-0.127500,51.507222
2,http://www.wikidata.org/entity/Q175106,Тим Хортонс,кофе,Канада,Оквилл,Point(-79.683333333 43.45),1964.0,-79.683333,43.450000



📈 Статистика по числовым столбцам:


,foundingYearShort,lon,lat
count,240.00,218.00,218.00
mean,1942.00,1.06,36.08
std,72.86,74.23,25.19
min,1600.00,-123.11,-34.90
25%,1910.00,-71.01,33.70
50%,1971.50,4.88,45.45
75%,1996.00,29.98,51.71
max,2022.00,153.40,66.42



🏷️ Категориальные столбцы:
• company_url: 189 уникальных
• company: 189 уникальных
• productType: 2 уникальных
• country: 48 уникальных
• hq: 129 уникальных
• hqCoord: 134 уникальных

📊 💰 df_fin: финансовые показатели
Размер: 1555 строк × 11 столбцов
Столбцы: company_url, company, productType, revenue, revenueYear, revenueCurrency, employees, employeesYear, netProfit, netProfitYear, netProfitCurrency

📋 Первые 3 строки:


,company_url,company,productType,revenue,revenueYear,revenueCurrency,employees,employeesYear,netProfit,netProfitYear,netProfitCurrency
0,http://www.wikidata.org/entity/Q101249024,Q101249024,чай,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,http://www.wikidata.org/entity/Q1020913,Bünting-Gruppe,чай,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,http://www.wikidata.org/entity/Q104530260,Tjiboeni-Tjipongpok Caoutchouc Maatschappij,кофе,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



📈 Статистика по числовым столбцам:


,revenue,revenueYear,employees,employeesYear,netProfit,netProfitYear
count,1.318000e+03,1318.00,1326.00,1323.00,1.310000e+03,1309.00
mean,2.152226e+10,2017.02,239374.16,2017.76,2.066543e+09,2015.29
std,9.209741e+09,5.47,102704.71,3.19,1.372751e+09,5.98
min,2.033000e+06,2007.00,18.00,2014.00,-1.480300e+07,2000.00
25%,1.329950e+10,2012.00,191000.00,2014.00,9.283000e+08,2011.00
50%,2.351800e+10,2018.00,245000.00,2016.00,1.856400e+09,2015.00
75%,2.766100e+10,2022.00,383000.00,2021.00,3.281600e+09,2020.00
max,3.718440e+10,2025.00,383000.00,2024.00,4.518300e+09,2025.00



🏷️ Категориальные столбцы:
• company_url: 234 уникальных
• company: 234 уникальных
• productType: 2 уникальных
• revenueCurrency: 5 уникальных
• netProfitCurrency: 4 уникальных

🔑 ПРОВЕРКА КЛЮЧЕЙ ДЛЯ ОБЪЕДИНЕНИЯ ТАБЛИЦ
• Уникальных company_url в df_info: 189
• Уникальных company_url в df_fin: 234
• Пересекающихся ключей: 189
✅ Ключи совпадают — таблицы можно объединить через pd.merge()


In [ ]:
# 🔍 Шаг 3. Обзор финансовых данных

def show_fin_info(df, name, n=5):
    """Расширенный обзор финансового DataFrame."""
    print(f"\n📊 {name}")
    print("=" * 70)
    print(f"Размер: {df.shape[0]} строк × {df.shape[1]} столбцов")
    print("Столбцы:", ", ".join(df.columns))

    print("\n📋 Первые строки:")
    display(df.head(n))

    # Статистика по числовым столбцам
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print("\n📈 Статистика по финансовым показателям:")
        display(df[numeric_cols].describe().round(2))

    # Категориальные столбцы
    print("\n🏷️ Категориальные данные:")
    for col in df.select_dtypes(include=['object']).columns:
        unique = df[col].nunique()
        top = df[col].mode()[0] if not df[col].mode().empty else "N/A"
        print(f"• {col}: {unique} уникальных | чаще всего: '{top}'")

# Запуск обзора
show_fin_info(df_fin, "Финансовые показатели чайно-кофейных компаний")

# ============================================================
# 💡 Дополнительные быстрые срезы
# ============================================================

print("\n" + "=" * 70)
print("📌 БЫСТРЫЕ СРЕЗЫ ПО ДАННЫМ")
print("=" * 70)

# Продукты
if "productType" in df_fin.columns:
    print("\n☕ Распределение по типам продуктов:")
    print(df_fin["productType"].value_counts())

# Валюты
currency_cols = ["revenueCurrency", "netProfitCurrency"]
for col in currency_cols:
    if col in df_fin.columns and df_fin[col].notna().any():
        print(f"\n💱 Уникальные валюты в {col}:")
        print(df_fin[col].dropna().unique())

# Компании с данными по выручке
if "revenue" in df_fin.columns and "company" in df_fin.columns:
    with_revenue = df_fin[df_fin["revenue"].notna()]
    if len(with_revenue) > 0:
        print(f"\n💰 Компании с данными по выручке ({len(with_revenue)} шт.):")
        display(with_revenue[["company", "revenue", "revenueYear", "revenueCurrency"]].head())

print("\n✅ Обзор завершён")


📊 Финансовые показатели чайно-кофейных компаний
Размер: 1555 строк × 10 столбцов
Столбцы: company, productType, revenue, revenueYear, revenueCurrency, employees, employeesYear, netProfit, netProfitYear, netProfitCurrency

📋 Первые строки:


,company,productType,revenue,revenueYear,revenueCurrency,employees,employeesYear,netProfit,netProfitYear,netProfitCurrency
0,Q101249024,чай,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Bünting-Gruppe,чай,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Tjiboeni-Tjipongpok Caoutchouc Maatschappij,кофе,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Tjiboeni-Tjipongpok Caoutchouc Maatschappij,чай,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Sumatra Rubber Cultuur Maatschappij ""Serbadjadi""",кофе,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



📈 Статистика по финансовым показателям:


,revenue,revenueYear,employees,employeesYear,netProfit,netProfitYear
count,1.318000e+03,1318.00,1326.00,1323.00,1.310000e+03,1309.00
mean,2.152226e+10,2017.02,239374.16,2017.76,2.066543e+09,2015.29
std,9.209741e+09,5.47,102704.71,3.19,1.372751e+09,5.98
min,2.033000e+06,2007.00,18.00,2014.00,-1.480300e+07,2000.00
25%,1.329950e+10,2012.00,191000.00,2014.00,9.283000e+08,2011.00
50%,2.351800e+10,2018.00,245000.00,2016.00,1.856400e+09,2015.00
75%,2.766100e+10,2022.00,383000.00,2021.00,3.281600e+09,2020.00
max,3.718440e+10,2025.00,383000.00,2024.00,4.518300e+09,2025.00



🏷️ Категориальные данные:
• company: 234 уникальных | чаще всего: 'Starbucks'
• productType: 2 уникальных | чаще всего: 'кофе'
• revenueCurrency: 5 уникальных | чаще всего: 'доллар США'
• netProfitCurrency: 4 уникальных | чаще всего: 'доллар США'

📌 БЫСТРЫЕ СРЕЗЫ ПО ДАННЫМ

☕ Распределение по типам продуктов:
productType
кофе    1457
чай       98
Name: count, dtype: int64

💱 Уникальные валюты в revenueCurrency:
['евро' 'чешская крона' 'Белорусский рубль' 'датская крона' 'доллар США']

💱 Уникальные валюты в netProfitCurrency:
['чешская крона' 'Белорусский рубль' 'евро' 'доллар США']

💰 Компании с данными по выручке (1318 шт.):


,company,revenue,revenueYear,revenueCurrency
34,Devolli Corporation,400000000.0,2021.0,евро
50,Jemča,152244000.0,2019.0,чешская крона
73,Белкофе,2033000.0,2023.0,Белорусский рубль
92,Mikel Coffee Company,33439000.0,2014.0,евро
93,Mikel Coffee Company,33439000.0,2014.0,евро



✅ Обзор завершён


## 🌍🏆 [4] Анализ: география и финансы

### 🌍 Географический рейтинг (по `df_info`):
- Топ-10 стран по количеству компаний;
- Распределение по типам продукта внутри стран;
- **Бонус**: координаты `lon`/`lat` готовы для построения карты.

### 💰 Финансовый рейтинг (по `df_fin`):
- Топ-10 компаний по выручке (с учётом дубликатов: берём последний год);
- Сравнение средних показателей по продуктам.

### ⚠️ Важное предупреждение о данных:
В `tea_coffee_fin.csv` наблюдается **сильная диспропорция**:
- 🟤 Кофейные компании: ~1457 строк
- 🟢 Чайные компании: ~98 строк

Это означает, что финансовые данные в Викиданных заполнены преимущественно для кофейного бизнеса. **Сравнение чая и кофе по финансовым показателям будет статистически ненадёжным.** В этом ноутбуке мы:
1. Честно отразим эту диспропорцию в выводах;
2. Проведём финансовый анализ в первую очередь для кофейных компаний;
3. Сохраним все данные для будущего обогащения, когда чайных компаний станет больше.



In [9]:
# 🌍🏆 Шаг 4. Географический и финансовый анализ

print("🌍🏆 АНАЛИЗ: География + Финансы")
print("=" * 70)

# ============================================================
# 🌍 1) Географический рейтинг (по df_info)
# ============================================================

print("\n🌍 ТОП-10 СТРАН ПО КОЛИЧЕСТВУ КОМПАНИЙ")
print("-" * 70)

if "country" in df_info.columns:
    country_stats = df_info["country"].value_counts().head(10)
    for country, count in country_stats.items():
        pct = count / len(df_info) * 100
        bar = "█" * int(pct / 2)
        print(f"{country:<40} {count:>3} | {pct:>5.1f}% {bar}")

# 🗺️ Бонус: проверка координат для карты
if "lon" in df_info.columns and "lat" in df_info.columns:
    valid_coords = df_info[["lon", "lat"]].notna().all(axis=1).sum()
    print(f"\n✅ Координаты для карты: {valid_coords} из {len(df_info)} компаний")

# ============================================================
# ☕🆚🫖 2) Распределение по продуктам (в обоих файлах)
# ============================================================

print("\n\n☕🆚🫖 РАСПРЕДЕЛЕНИЕ: ЧАЙ vs КОФЕ")
print("-" * 70)

for name, df in [("df_info", df_info), ("df_fin", df_fin)]:
    if "productType" in df.columns:
        print(f"\n📊 {name}:")
        print(df["productType"].value_counts())

# ============================================================
# 💰 3) Финансовый рейтинг (по df_fin_revenue — с обработкой дубликатов)
# ============================================================

print("\n\n💰 ТОП-10 КОМПАНИЙ ПО ВЫРУЧКЕ (последний доступный год)")
print("-" * 70)

if "revenue" in df_fin_revenue.columns and "company" in df_fin_revenue.columns:
    # Берём только строки с выручкой
    rev_valid = df_fin_revenue[df_fin_revenue["revenue"].notna()]

    if len(rev_valid) > 0:
        top_rev = rev_valid.nlargest(10, "revenue")[["company", "revenue", "revenueYear", "revenueCurrency", "productType"]]

        for i, (_, row) in enumerate(top_rev.iterrows(), 1):
            currency = row["revenueCurrency"] if pd.notna(row["revenueCurrency"]) else "N/A"
            year = int(row["revenueYear"]) if pd.notna(row["revenueYear"]) else "N/A"
            product = row["productType"] if pd.notna(row["productType"]) else "N/A"
            print(f"{i:>2}. {row['company']:<35} {row['revenue']:>12,.0f} {currency} ({year}) | {product}")
    else:
        print("⚠️  Нет данных по выручке для формирования рейтинга")

# ============================================================
# 📊 4) Честный отчёт о диспропорции
# ============================================================

print("\n\n⚠️  ОТЧЁТ О КАЧЕСТВЕ ДАННЫХ")
print("-" * 70)

if "productType" in df_fin.columns:
    tea_fin = (df_fin["productType"] == "чай").sum()
    coffee_fin = (df_fin["productType"] == "кофе").sum()

    print(f"• Чайных компаний с финансовыми данными: {tea_fin}")
    print(f"• Кофейных компаний с финансовыми данными: {coffee_fin}")

    if tea_fin < 20:
        print(f"\n🔴 ВНИМАНИЕ: данных о чайных компаниях недостаточно для статистического сравнения.")
        print("   Выводы по финансам релевантны преимущественно для кофейного сектора.")

🌍🏆 АНАЛИЗ: География + Финансы

🌍 ТОП-10 СТРАН ПО КОЛИЧЕСТВУ КОМПАНИЙ
----------------------------------------------------------------------
США                                       35 |  14.6% ███████
Нидерланды                                24 |  10.0% █████
Великобритания                            14 |   5.8% ██
Соединённое королевство Великобритании и Ирландии  10 |   4.2% ██
Канада                                    10 |   4.2% ██
Индия                                      7 |   2.9% █
Дания                                      7 |   2.9% █
Франция                                    6 |   2.5% █
Италия                                     6 |   2.5% █
Австралия                                  6 |   2.5% █

✅ Координаты для карты: 218 из 240 компаний


☕🆚🫖 РАСПРЕДЕЛЕНИЕ: ЧАЙ vs КОФЕ
----------------------------------------------------------------------

📊 df_info:
productType
кофе    149
чай      91
Name: count, dtype: int64

📊 df_fin:
productType
кофе    1457
чай       98
Name:

In [ ]:
# 🏆 Шаг 4. Финансовый рейтинг компаний

import pandas as pd

print("🏆 ФИНАНСОВЫЙ РЕЙТИНГ КОМПАНИЙ")
print("=" * 70)

# ============================================================
# 🔍 0. Проверка наличия необходимых столбцов
# ============================================================

required = ["company", "productType"]
financial = ["revenue", "employees", "netProfit"]

for col in required:
    if col not in df_fin.columns:
        raise ValueError(f"❌ Обязательный столбец '{col}' не найден!")

print(f"✅ Все необходимые столбцы присутствуют")
print(f"📋 Всего компаний в датасете: {len(df_fin)}\n")

# ============================================================
# 📊 1. Заполненность финансовых показателей
# ============================================================

print("📊 ЗАПОЛНЕННОСТЬ ФИНАНСОВЫХ ДАННЫХ")
print("-" * 70)
for col in financial:
    if col in df_fin.columns:
        filled = df_fin[col].notna().sum()
        print(f"{col:<12}: {filled:>3} из {len(df_fin)} компаний ({filled/len(df_fin)*100:.1f}%)")

# ============================================================
# 💰 2. Топ компаний по выручке
# ============================================================

if "revenue" in df_fin.columns:
    print("\n\n💰 ТОП-10 КОМПАНИЙ ПО ВЫРУЧКЕ")
    print("-" * 70)

    rev_df = df_fin[df_fin["revenue"].notna()].copy()

    if len(rev_df) > 0:
        top_rev = rev_df.nlargest(10, "revenue")[["company", "revenue", "revenueYear", "revenueCurrency", "productType"]]
        for i, (_, row) in enumerate(top_rev.iterrows(), 1):
            currency = row["revenueCurrency"] if pd.notna(row["revenueCurrency"]) else "N/A"
            year = int(row["revenueYear"]) if pd.notna(row["revenueYear"]) else "N/A"
            print(f"{i:>2}. {row['company']:<40} {row['revenue']:>15,.0f} {currency} ({year})")
    else:
        print("⚠️  Нет данных по выручке для формирования рейтинга")

# ============================================================
# 👥 3. Топ компаний по количеству сотрудников
# ============================================================

if "employees" in df_fin.columns:
    print("\n\n👥 ТОП-10 КОМПАНИЙ ПО КОЛИЧЕСТВУ СОТРУДНИКОВ")
    print("-" * 70)

    emp_df = df_fin[df_fin["employees"].notna()].copy()

    if len(emp_df) > 0:
        top_emp = emp_df.nlargest(10, "employees")[["company", "employees", "employeesYear", "productType"]]
        for i, (_, row) in enumerate(top_emp.iterrows(), 1):
            year = int(row["employeesYear"]) if pd.notna(row["employeesYear"]) else "N/A"
            print(f"{i:>2}. {row['company']:<40} {row['employees']:>10,.0f} сотрудников ({year})")
    else:
        print("⚠️  Нет данных по сотрудникам для формирования рейтинга")

# ============================================================
# 📈 4. Топ компаний по чистой прибыли
# ============================================================

if "netProfit" in df_fin.columns:
    print("\n\n📈 ТОП-10 КОМПАНИЙ ПО ЧИСТОЙ ПРИБЫЛИ")
    print("-" * 70)

    profit_df = df_fin[df_fin["netProfit"].notna()].copy()

    if len(profit_df) > 0:
        top_profit = profit_df.nlargest(10, "netProfit")[["company", "netProfit", "netProfitYear", "netProfitCurrency", "productType"]]
        for i, (_, row) in enumerate(top_profit.iterrows(), 1):
            currency = row["netProfitCurrency"] if pd.notna(row["netProfitCurrency"]) else "N/A"
            year = int(row["netProfitYear"]) if pd.notna(row["netProfitYear"]) else "N/A"
            print(f"{i:>2}. {row['company']:<40} {row['netProfit']:>15,.0f} {currency} ({year})")
    else:
        print("⚠️  Нет данных по прибыли для формирования рейтинга")

# ============================================================
# ☕🆚🫖 5. Сравнение чая и кофе (если есть productType)
# ============================================================

if "productType" in df_fin.columns:
    print("\n\n☕🆚🫖 СРАВНЕНИЕ: ЧАЙ vs КОФЕ")
    print("-" * 70)

    comparison = {}
    for metric in ["revenue", "employees", "netProfit"]:
        if metric in df_fin.columns:
            comp = df_fin.groupby("productType")[metric].agg(["count", "mean", "median", "max"])
            comp = comp.rename(columns={"count": "записей", "mean": "среднее", "median": "медиана", "max": "максимум"})
            print(f"\n{metric.upper()}:")
            print(comp.round(2).to_markdown())

print("\n\n✅ Финансовый рейтинг завершён")

🏆 ФИНАНСОВЫЙ РЕЙТИНГ КОМПАНИЙ
✅ Все необходимые столбцы присутствуют
📋 Всего компаний в датасете: 1555

📊 ЗАПОЛНЕННОСТЬ ФИНАНСОВЫХ ДАННЫХ
----------------------------------------------------------------------
revenue     : 1318 из 1555 компаний (84.8%)
employees   : 1326 из 1555 компаний (85.3%)
netProfit   : 1310 из 1555 компаний (84.2%)


💰 ТОП-10 КОМПАНИЙ ПО ВЫРУЧКЕ
----------------------------------------------------------------------
 1. Starbucks                                 37,184,400,000 доллар США (2025)
 2. Starbucks                                 37,184,400,000 доллар США (2025)
 3. Starbucks                                 37,184,400,000 доллар США (2025)
 4. Starbucks                                 37,184,400,000 доллар США (2025)
 5. Starbucks                                 37,184,400,000 доллар США (2025)
 6. Starbucks                                 37,184,400,000 доллар США (2025)
 7. Starbucks                                 37,184,400,000 доллар США (2025)
 8. 

 Ячейка 10: Код — Сохранение результатов (❗НОВОЕ)
✅ Что добавляем: Сохранение очищенных DataFrame в CSV

In [10]:
# 💾 Шаг 5. Сохранение очищенных данных

print("💾 СОХРАНЕНИЕ РЕЗУЛЬТАТОВ")
print("=" * 70)

# Сохраняем информационный файл
info_path = "data/tea_coffee_info_cleaned.csv"
df_info.to_csv(info_path, index=False, encoding="utf-8-sig")
print(f"✅ df_info сохранён: {info_path}")

# Сохраняем финансовый файл
fin_path = "data/tea_coffee_fin_cleaned.csv"
df_fin.to_csv(fin_path, index=False, encoding="utf-8-sig")
print(f"✅ df_fin сохранён: {fin_path}")

# Сохраняем рейтинг по выручке (для удобства)
if "df_fin_revenue" in locals():
    rev_path = "data/tea_coffee_revenue_ranking.csv"
    df_fin_revenue.to_csv(rev_path, index=False, encoding="utf-8-sig")
    print(f"✅ Рейтинг по выручке сохранён: {rev_path}")

print("\n🎉 Все очищенные данные сохранены в папке data/")

💾 СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
✅ df_info сохранён: data/tea_coffee_info_cleaned.csv
✅ df_fin сохранён: data/tea_coffee_fin_cleaned.csv
✅ Рейтинг по выручке сохранён: data/tea_coffee_revenue_ranking.csv

🎉 Все очищенные данные сохранены в папке data/


### 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий `python-ai-Gailunaite-Darya` в Google Colab
- ✅ Прочитали два CSV-файла:
  - 🫖 `tea_coffee_info.csv` → `df_info`: компании, страны, координаты, годы основания
  - 💰 `tea_coffee_fin.csv` → `df_fin`: выручка, сотрудники, прибыль
- ✅ Очистили и нормализовали данные:
  - 🔑 Переименовали `company` → `company_url` (сохранили ключ для будущего `merge`!)
  - 🏷️ Убрали суффикс `*Label` у читаемых названий
  - 🗺️ Распарсили `hqCoord` → числовые `lon`/`lat` для карт
  - 🔢 Привели годы и финансы к числовому типу
  - 🚫 Отфильтровали некорректные значения (`foundingYearShort ≤ 0`)
  - 🔄 Обработали дубликаты в финансовых данных (взяли последний год по компании)
- ✅ Выполнили разведочный анализ:
  - 🌍 Географический рейтинг: топ стран по количеству компаний
  - ☕🆚🫖 Честно отразили диспропорцию: чайных компаний с финансами ~98, кофейных ~1457
  - 💰 Финансовый рейтинг: топ-10 по выручке (с учётом дубликатов)
  - 🔑 Проверили совпадение ключей `company_url` для будущего объединения таблиц
- ✅ Сохранили результаты:
  - `data/tea_coffee_info_cleaned.csv`
  - `data/tea_coffee_fin_cleaned.csv`
  - `data/tea_coffee_revenue_ranking.csv`

---

### 🎯 Ключевые выводы:

| Вопрос | Ответ |
|--------|-------|
| 🔑 Сохранён ли ключ для объединения таблиц? | ✅ Да, `company_url` в обоих DataFrame |
| 🗺️ Готовы ли координаты для карты? | ✅ Да, добавлены столбцы `lon`, `lat` |
| 📅 Корректны ли годы основания? | ✅ Да, отфильтрованы значения ≤ 0 |
| 💰 Можно ли сравнивать чай и кофе по финансам? | 🔴 Нет, данных о чае слишком мало (~98 строк против ~1457) |
| 🏆 Кто лидирует по выручке? | См. рейтинг выше (часто это крупные кофейные сети) |

---

### 🚀 Что дальше? (Week 3)

1. **Визуализация карты**: `plotly.express.scatter_geo` по `lon`/`lat`
2. **Объединение таблиц**: `pd.merge(df_info, df_fin, on="company_url")`
3. **Динамика выручки**: линейный график для компаний с данными за несколько лет
4. **Экспорт отчёта**: генерация Markdown/PDF с итогами анализа



| Вопрос | Ответ (пример) |
|--------|---------------|
| 🌍 Какие страны лидируют? | *Зависит от данных: например, Великобритания, Германия* |
| ☕ Чая или кофе больше? | *Сравните `df["productType"].value_counts()`* |
| 💰 У кого самая высокая выручка? | *Смотрите топ в ячейке финансового рейтинга* |
| 📅 Сколько компаний основано до 1800 года? | *Фильтр: `df[df["foundingYearShort"] < 1800]`* |
| ❓ Насколько полные финансовые данные? | *Проверьте отчёт о заполненности в ячейке очистки* |

> 💡 **Примечание**: Если какие-то рейтинги оказались пустыми — это не ошибка кода, а сигнал о том, что в Викиданных пока мало заполненных финансовых полей для этих компаний.

---

### 🚀 Что дальше? (Week 3 и далее)

В следующем ноутбуке мы сможем:

1. **Объединить два файла** (`info` + `fin`) по названию компании → получить единую таблицу `df_full`
2. **Углубить анализ**:
   - Группировка: «Средняя выручка по странам», «Динамика по десятилетиям»
   - Фильтрация: «Показать только чайные компании старше 100 лет»
   - Агрегация: «Общее количество сотрудников в индустрии»
3. **Построить визуализации** 🎨:
   - 🗺️ Карта штаб-квартир (по координатам `hqCoord`)
   - 📊 Столбчатая диаграмма: топ-10 компаний по выручке
   - 🥧 Круговая диаграмма: доля чая и кофе по странам
4. **Экспортировать результаты**:
   - Сохранить очищенные данные в `*_cleaned.csv`
   - Сгенерировать отчёт в Markdown или Excel

---

### 🧰 Полезные команды для самостоятельной работы:

```python
# 📋 Посмотреть уникальные значения
df_fin["productType"].unique()

# 🔍 Отфильтровать компании по условию
old_tea = df_info[(df_info["foundingYearShort"] < 1800) & (df_info["productType"] == "чай")]

# 📈 Быстрая статистика по группе
df_fin.groupby("productType")["revenue"].mean()

# 💾 Сохранить результат
df_fin.to_csv("data/tea_coffee_fin_cleaned.csv", index=False, encoding="utf-8-sig")